# Olist late-delivery risk — train on Colab

Runs the full pipeline on Colab hardware and exports a deployable model.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. XGBoost uses it; the
other models don't, but the CPU allocation is better on GPU runtimes anyway.

**Session limits.** Colab disconnects after ~90 minutes idle and reclaims everything
in `/content`. Section 2 puts the data and outputs on Google Drive so a disconnect
costs you a re-run, not the data.

## 1. Upload the code

In [ ]:
# Upload olist-delivery-risk.zip when prompted.
from google.colab import files
import zipfile, pathlib, shutil

uploaded = files.upload()
zip_name = next(iter(uploaded))

shutil.rmtree('/content/olist', ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

PROJECT = pathlib.Path('/content/olist')
assert (PROJECT / 'src' / 'olist' / 'train.py').exists(), \
    f'Unexpected layout. Found: {[p.name for p in PROJECT.iterdir()]}'
print('Code at', PROJECT)

In [ ]:
%cd /content/olist
!pip install -q -e ".[dev]"

import sklearn, xgboost, imblearn, pandas
print('pandas', pandas.__version__, '| sklearn', sklearn.__version__,
      '| xgboost', xgboost.__version__, '| imblearn', imblearn.__version__)

In [ ]:
# Confirm the code works on Colab's library versions before touching real data.
!python -m pytest -q

## 2. Get the data in

Two options. **Drive is strongly preferred** — the nine CSVs are ~120 MB, and a
browser upload has to be repeated every time the session dies.

Put the nine CSVs in a Drive folder (e.g. `MyDrive/olist_data/`), then run the
Drive cell. Use the browser-upload cell only as a fallback.

In [ ]:
# --- Option A: Google Drive (recommended) ---
from google.colab import drive
import pathlib, shutil

drive.mount('/content/drive')

DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/olist_data')   # <- edit if needed
RAW = pathlib.Path('/content/olist/data/raw')
RAW.mkdir(parents=True, exist_ok=True)

found = sorted(DRIVE_DATA.glob('*.csv'))
print(f'{len(found)} CSVs in {DRIVE_DATA}')
for src in found:
    shutil.copy(src, RAW / src.name)
    print('  copied', src.name)

In [ ]:
# --- Option B: browser upload (fallback; slow, and lost on disconnect) ---
# from google.colab import files
# import pathlib, shutil
# RAW = pathlib.Path('/content/olist/data/raw'); RAW.mkdir(parents=True, exist_ok=True)
# for name in files.upload():
#     shutil.move(name, RAW / name)

In [ ]:
# Preflight: all nine files, correct schemas, unique join keys.
!python -m olist.validate

## 3. Clean and merge

Takes a few minutes, mostly collapsing the million geolocation rows to zip centroids.
Expect ~96,470 modelling rows at an 8.11% late rate. If the order count after merging
is not 99,441, the run raises rather than continuing with duplicated rows.

In [ ]:
!python -m olist.validate --build

In [ ]:
# Back the processed table up to Drive so a disconnect doesn't cost the merge.
import shutil, pathlib

BACKUP = pathlib.Path('/content/drive/MyDrive/olist_outputs')
BACKUP.mkdir(parents=True, exist_ok=True)
src = pathlib.Path('/content/olist/data/processed/orders_features.parquet')
if src.exists():
    shutil.copy(src, BACKUP / src.name)
    print('Backed up to', BACKUP / src.name)

## 4. Baseline

Untuned, both scenarios. A couple of minutes each, and it tells you what tuning has
to beat. The `post_dispatch` run quantifies exactly what the carrier-handover signal
is worth — if it is much stronger, the right intervention point is escalation after
dispatch rather than prediction at order time, which is a business finding, not just
a modelling one.

In [ ]:
!python -m olist.train --scenario pre_dispatch

In [ ]:
!python -m olist.train --scenario post_dispatch

## 5. Tune

40 sampled configurations × 5 folds × 3 models, with SMOTE refitted inside every
fold. **Roughly 45–90 minutes on Colab.** Keep the tab open; Colab kills idle sessions.

To shorten it, tune one model: `--models xgboost`.

In [ ]:
!python -m olist.train --scenario pre_dispatch --tune

## 6. Inspect the results

In [ ]:
import pandas as pd, json, pathlib

REPORTS = pathlib.Path('/content/olist/reports')

metrics = pd.read_csv(REPORTS / 'metrics_pre_dispatch.csv', index_col=0)
cols = ['roc_auc', 'average_precision', 'chosen_threshold', 'recall_at_chosen',
        'precision_at_chosen', 'flag_rate', 'cost_per_order',
        'recall_at_10pct_budget', 'recall_at_20pct_budget']
display(metrics[[c for c in cols if c in metrics.columns]].round(4))

run = json.loads((REPORTS / 'run_pre_dispatch.json').read_text())
print('Selected:', run['selected'])
print('Threshold:', round(run['threshold']['threshold'], 4))
for name, info in run.get('search', {}).items():
    if info:
        print(f"\n{name}: CV score {info['best_cv_score']:.4f}")
        for k, v in info['best_params'].items():
            print(f'    {k} = {v}')

In [ ]:
# Permutation importance on the held-out set: the honest ranking.
imp = pd.read_csv(REPORTS / 'permutation_importance_pre_dispatch.csv')
top = imp.head(20).iloc[::-1]

ax = top.plot.barh(x='feature', y='importance_mean', xerr='importance_std',
                   figsize=(9, 8), legend=False, color='#378ADD')
ax.set_xlabel('Drop in average precision when shuffled')
ax.set_title('Permutation importance (test set)')
ax.figure.tight_layout()

In [ ]:
# Expected cost against threshold. The minimum is the chosen operating point;
# the flat line is the do-nothing baseline. If the curve never dips below it,
# the model is not worth deploying at these cost assumptions.
import matplotlib.pyplot as plt

curve = pd.read_csv(REPORTS / 'cost_curve_pre_dispatch.csv')
chosen = run['threshold']

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(curve['threshold'], curve['expected_cost'], color='#378ADD')
ax.axvline(chosen['threshold'], color='#D85A30', ls='--',
           label=f"chosen = {chosen['threshold']:.3f}")
ax.axhline(chosen['baseline_cost'], color='grey', ls=':', label='do nothing')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Expected cost')
ax.set_title('Cost-optimal operating point')
ax.legend()
fig.tight_layout()

## 7. Export for deployment

One artefact: preprocessing, resampling, model and threshold in a single pickle.
Nothing else has to be kept in sync at serving time.

**Pickles are version-sensitive.** Loading in an environment with a different
scikit-learn can fail or, worse, load with altered behaviour. The versions are
recorded below — pin them wherever you deploy.

In [ ]:
import joblib, json, sklearn, xgboost, imblearn, pandas as pd, numpy as np, pathlib

MODELS = pathlib.Path('/content/olist/models')
artefact = joblib.load(MODELS / 'model_pre_dispatch.joblib')

artefact['environment'] = {
    'python': __import__('sys').version.split()[0],
    'scikit-learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
    'imbalanced-learn': imblearn.__version__,
    'pandas': pd.__version__,
    'numpy': np.__version__,
}
joblib.dump(artefact, MODELS / 'model_pre_dispatch.joblib')

print('Model     :', artefact['model_name'])
print('Scenario  :', artefact['scenario'])
print('Threshold :', round(artefact['threshold'], 4))
print('Features  :', len(artefact['feature_columns']))
print(json.dumps(artefact['environment'], indent=2))

In [ ]:
# Round-trip check: reload and score real rows. Catches a corrupt or
# unloadable pickle here rather than in deployment.
import pandas as pd, joblib, pathlib

reloaded = joblib.load(pathlib.Path('/content/olist/models/model_pre_dispatch.joblib'))
df = pd.read_parquet('/content/olist/data/processed/orders_features.parquet')
sample = df[reloaded['feature_columns']].tail(500)

proba = reloaded['pipeline'].predict_proba(sample)[:, 1]
flagged = (proba >= reloaded['threshold']).sum()
print(f'Scored {len(proba)} orders | mean risk {proba.mean():.3f} | '
      f'flagged {flagged} ({flagged / len(proba):.1%})')

In [ ]:
# Bundle model + reports and save to Drive, then download.
import shutil, pathlib
from google.colab import files

BUNDLE = pathlib.Path('/content/olist_model_bundle')
shutil.rmtree(BUNDLE, ignore_errors=True)
(BUNDLE / 'models').mkdir(parents=True)
(BUNDLE / 'reports').mkdir(parents=True)

for p in pathlib.Path('/content/olist/models').glob('*.joblib'):
    shutil.copy(p, BUNDLE / 'models' / p.name)
for p in pathlib.Path('/content/olist/reports').glob('*.*'):
    shutil.copy(p, BUNDLE / 'reports' / p.name)

archive = shutil.make_archive('/content/olist_model_bundle', 'zip', BUNDLE)

drive_out = pathlib.Path('/content/drive/MyDrive/olist_outputs')
if drive_out.parent.exists():
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive, drive_out / 'olist_model_bundle.zip')
    print('Saved to Drive:', drive_out / 'olist_model_bundle.zip')

files.download(archive)

## 8. Serving it locally

Unzip the bundle into your project so `models/model_pre_dispatch.joblib` sits where
the app expects it, then:

```bash
streamlit run app/streamlit_app.py
```

To score in your own code:

```python
import joblib, pandas as pd

art = joblib.load('models/model_pre_dispatch.joblib')
X = pd.DataFrame([order_dict])[art['feature_columns']]   # missing cols -> imputed
risk = art['pipeline'].predict_proba(X)[0, 1]
flag = risk >= art['threshold']
```

**Before this is worth deploying, know what it is.** At an 8% base rate and
~0.72 AUC, the value is in ranking a limited alert budget, not in per-order
certainty. Chasing the riskiest 20% of orders catches roughly 43% of late
deliveries; the score for any individual order is weak evidence. The cost figures
in `config.CostConfig` are placeholders — substitute real Olist numbers before
quoting a saving, because they determine the threshold and therefore every
operational consequence of the model.